# Step 1: Tokenization (with BPE)

<!-- notebook-only -->

> **How to use this notebook**
> - **Just reading?** The text cells contain the whole lesson. You can skip every code cell. An even nicer way to read: the website, built with `.venv/bin/python website/build.py --serve`.
> - **Want to practice?** Run the code cells from top to bottom (Shift+Enter). They rebuild the tokenizer and print every step. In **Your turn** cells, replace `None` with your answer: `check(...)` says if it is right and gives a hint if not, and `solution("1.1")` shows the worked answer.
> - Every question also has a **Show the answer** box you can open.
> - Near the end, a **playground** builds the tokenizer from your own sentences.

<!-- /notebook-only -->

In [ ]:
# Run this first (practice only): it makes the `toddler` package (repo root) importable
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

from toddler.data import CORPUS
from toddler.exercises import check, solution

## 1.1 Why tokenize?

**Goal:** turn text into a list of whole numbers, and turn numbers back into text. The model only does math and never sees letters.

The tokenizer is not part of the "brain". It is built *before* the model and never changes after that.

## 1.2 The simplest way: one word = one token

**Encode:** split the text at spaces, then look up each piece in a table:

| ID | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| word | `.` | I | am | want | hungry | sad | tired | milk | mommy | bed |

```
text:  I   am  hungry  .   I   want  milk  .
IDs:   1   2   4       0   1   3     7     0
```

**Decode:** look up the other way. `[1, 3, 8, 0]` → "I want mommy ."

This simple word table is only used in sections 1.2 and 1.3. From section 1.5 on, the toddler uses BPE, and the IDs change.

## 1.3 Problem: words it has never heard

"I am sleepy ." → "sleepy" isn't in the table → `[1, 2, ?, 0]`.

You could put in a special `<unk>` ("unknown") token, but then the meaning is lost. Real text has no end of new words: names, typos ("hungryyy"), other languages, code, emoji. You can't list them all.

## 1.4 The other extreme: one letter = one token

No word is ever unknown, and the vocabulary is tiny. But:
- "I am hungry . I want milk ." becomes **27** tokens instead of **8**.
- One letter means almost nothing, so the model wastes effort putting words back together.
- In attention, every token looks at every earlier token, so the work grows with length × length. Here that's about 10× more work.

## 1.5 The middle way real LLMs use: BPE (byte pair encoding)

BPE **learns** its pieces from the training text. Frequent words become one token, and rare words are cut into known pieces.

**How to read the examples:** `_` means "a new word starts here" (there was a space before it). Inside the grey boxes, a normal space separates pieces: `_ w a n t` is 5 pieces, `_want` is 1 piece.

### Part A: Prepare the text

**A1. Split the text into words and put `_` in front of every word.** Later, this tells us where to put the spaces back.

```
I am hungry . I want milk .   →   _I _am _hungry _. _I _want _milk _.
```

Do the same for the other two sentences.

**A2. Count how often each word appears** (3 sentences = 24 words):

| word | `_I` | `_.` | `_am` | `_want` | `_hungry` | `_sad` | `_tired` | `_milk` | `_mommy` | `_bed` |
|---|---|---|---|---|---|---|---|---|---|---|
| times | 6 | 6 | 3 | 3 | 1 | 1 | 1 | 1 | 1 | 1 |

**A3. Cut every word into single symbols:**

```
_I    → _ I            _hungry → _ h u n g r y      _milk  → _ m i l k
_.    → _ .            _sad    → _ s a d            _mommy → _ m o m m y
_am   → _ a m          _tired  → _ t i r e d        _bed   → _ b e d
_want → _ w a n t
```

**A4. Starting vocabulary:** every different symbol, 21 in total:

```
_ I a m h u n g r y . w t i l k s d o e b
```

In [ ]:
# Try it: A1, put `_` in front of every word
def to_words(text):
    return ["_" + word for word in text.split()]

for sentence in CORPUS:
    print(f"{sentence:<30} → {' '.join(to_words(sentence))}")

In [ ]:
# Try it: A2, count the words
def count_words(corpus):
    counts = {}  # insertion order = order of first appearance in the text
    for text in corpus:
        for word in to_words(text):
            counts[word] = counts.get(word, 0) + 1
    return counts

word_counts = count_words(CORPUS)
for word, count in word_counts.items():
    print(f"{word:<8} ×{count}")
print("total:", sum(word_counts.values()), "words")

In [ ]:
# Try it: A3 and A4, cut into symbols and list the starting vocabulary
splits = {word: list(word) for word in word_counts}
for word, pieces in splits.items():
    print(f"{word:<8} → {' '.join(pieces)}")

start_vocab = []
for pieces in splits.values():
    for symbol in pieces:
        if symbol not in start_vocab:
            start_vocab.append(symbol)
print(f"\nstarting vocabulary, {len(start_vocab)} symbols: {' '.join(start_vocab)}")

### Part B: Training (the merge loop)

Repeat these 3 moves:
1. **Count** every pair of pieces that sit side by side inside a word. A word that appears 3 times counts 3 times.
2. **Pick** the pair with the highest count. If there's a tie, take the pair you meet first when reading the text from the start. Any fixed rule works; this one is ours.
3. **Glue** that pair into one new piece everywhere it appears. Add the new piece to the vocabulary and write the merge in the **merge list**.

![The BPE merge loop](figures/01_bpe_loop.png)

#### Round 1, in full detail

**Count:**

| pair | found in | count |
|---|---|---|
| `_ I` | `_I` ×6 | **6** |
| `_ .` | `_.` ×6 | **6** |
| `_ a`, `a m` | `_am` ×3 | 3 each |
| `_ w`, `w a`, `a n`, `n t` | `_want` ×3 | 3 each |
| `_ m` | `_milk`, `_mommy` | 2 |
| `e d` | `_tired`, `_bed` | 2 |
| every other pair (`h u`, `s a`, `m o`, …) | one word each | 1 |

**Pick:** `_ I` and `_ .` are tied at 6. `_I` comes first in the text, so `_ I` wins.

**Glue:** every `_ I` becomes `_I`. The vocabulary now has 22 pieces.

In [ ]:
# Try it: move 1, count every side-by-side pair
def count_pairs(splits, word_counts):
    pairs = {}
    for word, pieces in splits.items():
        for pair in zip(pieces, pieces[1:]):
            pairs[pair] = pairs.get(pair, 0) + word_counts[word]
    return pairs

pairs = count_pairs(splits, word_counts)
for (left, right), count in sorted(pairs.items(), key=lambda item: -item[1]):  # biggest first; ties keep text order
    print(f"'{left} {right}'  {count}")

In [ ]:
# Try it: move 2, pick the winner
best = max(pairs, key=pairs.get)  # on a tie, max() keeps the first pair met, and `pairs` is in text order
print("winner:", best, "with count", pairs[best])

In [ ]:
# Try it: move 3, glue the winner everywhere
def merge_pair(pair, pieces):
    left, right = pair
    out, i = [], 0
    while i < len(pieces):
        if i + 1 < len(pieces) and (pieces[i], pieces[i + 1]) == pair:
            out.append(left + right)
            i += 2
        else:
            out.append(pieces[i])
            i += 1
    return out

for word, pieces in splits.items():
    merged = merge_pair(best, pieces)
    if merged != pieces:
        print(f"{' '.join(pieces)}  →  {' '.join(merged)}")

#### Rounds 1 to 10 (the same 3 moves each time, counting again after each glue)

| merge | top pairs right now (count) | winner | what changes |
|---|---|---|---|
| #1 | `_ I` 6, `_ .` 6 | `_ I` | `_ I` → `_I` |
| #2 | `_ .` 6 | `_ .` | `_ .` → `_.` |
| #3 | `_ a`, `a m`, `_ w`, `w a`, `a n`, `n t` (3 each) | `_ a` (first in text) | `_ a m` → `_a m` |
| #4 | `_a m`, `_ w`, `w a`, `a n`, `n t` (3 each) | `_a m` | `_a m` → `_am` |
| #5 | `_ w`, `w a`, `a n`, `n t` (3 each) | `_ w` | `_ w a n t` → `_w a n t` |
| #6 | `_w a`, `a n`, `n t` (3 each) | `_w a` | `_w a n t` → `_wa n t` |
| #7 | `_wa n`, `n t` (3 each) | `_wa n` | `_wa n t` → `_wan t` |
| #8 | `_wan t` 3 | `_wan t` | `_wan t` → `_want` |
| #9 | `_ m` 2, `e d` 2 | `_ m` (milk is in sentence 1) | `_ m i l k` → `_m i l k`, `_ m o m m y` → `_m o m m y` |
| #10 | `e d` 2 | `e d` | `_ t i r e d` → `_ t i r ed`, `_ b e d` → `_ b ed` |

Two things to notice:
- A merge only glues that **exact** pair. At #3, `_want` is not changed, because its `a` comes after `w`, not after `_`.
- The counts are **done again after every glue**. After #3, the pair `a m` no longer exists; it has become `_a m`.

**How the words look after 10 merges:**

```
_I     (×6)  done           _hungry → _ h u n g r y
_.     (×6)  done           _sad    → _ s a d
_am    (×3)  done           _tired  → _ t i r ed
_want  (×3)  done           _milk   → _m i l k
                            _mommy  → _m o m m y
                            _bed    → _ b ed
```

The common words are already one piece. Every pair still left appears **only once**.

#### Rounds 11 to 32

The rule stays the same. Every count is now 1, so every round is a tie. "First in text" then simply finishes the rare words, from left to right:

| merges | new pieces |
|---|---|
| #11–#16 | `_h` `_hu` `_hun` `_hung` `_hungr` `_hungry` |
| #17–#19 | `_mi` `_mil` `_milk` |
| #20–#22 | `_s` `_sa` `_sad` |
| #23–#26 | `_mo` `_mom` `_momm` `_mommy` |
| #27–#30 | `_t` `_ti` `_tir` `_tired` |
| #31–#32 | `_b` `_bed` |

After #32, every word is one piece and no pairs are left, so we **stop**. Real tokenizers stop earlier, when the vocabulary reaches a chosen size such as 50,000.

In [ ]:
# Try it: the whole training loop, one line per round
def train_bpe(corpus, max_merges=None, verbose=False):
    word_counts = count_words(corpus)
    splits = {word: list(word) for word in word_counts}
    vocab = []
    for pieces in splits.values():
        for symbol in pieces:
            if symbol not in vocab:
                vocab.append(symbol)
    merges = []
    while max_merges is None or len(merges) < max_merges:
        pairs = count_pairs(splits, word_counts)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        new_splits = {word: merge_pair(best, pieces) for word, pieces in splits.items()}
        if verbose:
            top = [f"{a} {b}" for (a, b), count in pairs.items() if count == pairs[best]]
            top = ", ".join(top[:4]) + (", ..." if len(top) > 4 else "")
            winner = f"{best[0]} {best[1]}"
            changes = ";  ".join(f"{' '.join(splits[w])} → {' '.join(new_splits[w])}"
                                 for w in splits if new_splits[w] != splits[w])
            print(f"#{len(merges) + 1:<3} top ({pairs[best]}): {top:<28} winner: {winner:<12} | {changes}")
        splits = new_splits
        merges.append(best)
        vocab.append(best[0] + best[1])
    return merges, vocab

merges, vocab = train_bpe(CORPUS, verbose=True)

#### What training gives you: two things

**1. The vocabulary.** It lists the 21 symbols first, then the 32 merges in order. The number in front of each piece is its ID:

```
 0 _    1 I    2 a    3 m    4 h    5 u    6 n    7 g    8 r    9 y   10 .
11 w   12 t   13 i   14 l   15 k   16 s   17 d   18 o   19 e   20 b
21 _I      22 _.      23 _a      24 _am     25 _w      26 _wa     27 _wan    28 _want
29 _m      30 ed      31 _h      32 _hu     33 _hun    34 _hung   35 _hungr  36 _hungry
37 _mi     38 _mil    39 _milk   40 _s      41 _sa     42 _sad    43 _mo     44 _mom
45 _momm   46 _mommy  47 _t      48 _ti     49 _tir    50 _tired  51 _b      52 _bed
```

**2. The merge list** (#1 to #32 above). This is the recipe that cuts any new text the same way every time.

In [ ]:
# Try it: the vocabulary with its IDs
token_to_id = {token: i for i, token in enumerate(vocab)}
print(f"{len(vocab)} tokens = {len(start_vocab)} symbols + {len(merges)} merges\n")
for token_id, token in enumerate(vocab):
    print(f"{token_id:>2} {token:<8}", end="\n" if token_id % 8 == 7 else "")
print()

### Part C: Using it (encoding)

The rule: put `_` in front of each word and cut it into symbols. Then go down the merge list in order (#1, #2, … #32). Wherever a merge's pair appears, glue it. At the end, look up the IDs.

**Example 1: "wanted", a word it has never seen**

| merge tried | found? | pieces after |
|---|---|---|
| (start) | | `_ w a n t e d` |
| #1–#4 (`_ I`, `_ .`, `_ a`, `_a m`) | no | `_ w a n t e d` |
| #5 `_ w` | yes | `_w a n t e d` |
| #6 `_w a` | yes | `_wa n t e d` |
| #7 `_wa n` | yes | `_wan t e d` |
| #8 `_wan t` | yes | `_want e d` |
| #9 `_ m` | no | `_want e d` |
| #10 `e d` | yes | `_want ed` |
| #11–#32 | no | `_want ed` |

Result: `_want` + `ed` → IDs **28, 30**. No "unknown" token is needed.

The same thing drawn as a tree: each merge glues two pieces into one, from the bottom up.

![How "wanted" is cut](figures/01_bpe_merge_tree.png)

**Example 2: "milky":** `_ m i l k y` → #9 → `_m i l k y` → #17 → `_mi l k y` → #18 → `_mil k y` → #19 → `_milk y` → IDs **39, 9**

**Example 3: "mom":** `_ m o m` → #9 → `_m o m` → #23 → `_mo m` → #24 → `_mom` → one token, ID **44**. A leftover piece of "mommy" turns out to be useful.

**Example 4: "sleepy":** `p` is not one of the 21 symbols, so the word can't be encoded at all. That's why real tokenizers start from the 256 possible **bytes**. Every letter of every language is made of bytes, so nothing can ever be unknown.

In [ ]:
# Try it: encode "wanted", showing every merge tried
def encode_word(word, merges, verbose=False, show_misses=False):
    pieces = list(word)
    if verbose:
        print(f"{'start':<16}{' '.join(pieces)}")
    for number, pair in enumerate(merges, start=1):
        merged = merge_pair(pair, pieces)
        found = merged != pieces
        if verbose and (found or show_misses):
            label = f"#{number} {pair[0]} + {pair[1]}"
            print(f"{label:<16}" + ("yes → " + " ".join(merged) if found else "no"))
        pieces = merged
    return pieces

pieces = encode_word("_wanted", merges, verbose=True, show_misses=True)
print("\nresult:", pieces, "→ IDs", [token_to_id[p] for p in pieces])

In [ ]:
# Try it: "milky" and "mom", showing only the merges that fire
for word in ["_milky", "_mom"]:
    print(f"--- {word}")
    pieces = encode_word(word, merges, verbose=True)
    print("result:", pieces, "→ IDs", [token_to_id[p] for p in pieces], "\n")

In [ ]:
# Try it: "sleepy" has a letter the toddler never heard
pieces = encode_word("_sleepy", merges, verbose=True)
print("\nnot in the vocabulary:", [p for p in pieces if p not in token_to_id])

### Part D: Decoding (IDs → text)

Look up each ID, glue the pieces together, turn each `_` back into a space, and remove the first space:

```
21 28 39 9 22  →  _I _want _milk y _.  →  "_I_want_milky_."  →  "I want milky ."
```

The `_` tells decoding where the spaces go: `_want` + `ed` becomes "wanted", not "want ed". GPT tokenizers write this mark as `Ġ` and SentencePiece writes it as `▁`; it's the same idea.

In [ ]:
# Try it: encode and decode whole sentences
def encode(text):
    return [token_to_id[piece] for word in to_words(text) for piece in encode_word(word, merges)]

def decode(ids):
    text = "".join(vocab[i] for i in ids).replace("_", " ")
    return text[1:] if text.startswith(" ") else text

for sentence in CORPUS:
    ids = encode(sentence)
    print(f"{sentence:<30} → {ids} → {decode(ids)!r}")

In [ ]:
# Try it: decoding, step by step
ids = [21, 28, 39, 9, 22]
pieces = [vocab[i] for i in ids]
glued = "".join(pieces)
print("IDs            ", ids)
print("pieces         ", pieces)
print("glued          ", repr(glued))
print("_ → space      ", repr(glued.replace("_", " ")))
print("drop 1st space ", repr(decode(ids)))

### Part E: The toddler's sentences, now as BPE tokens

```
I am hungry . I want milk .   →  21 24 36 22 21 28 39 22
I am sad . I want mommy .     →  21 24 42 22 21 28 46 22
I am tired . I want bed .     →  21 24 50 22 21 28 52 22
```

What this changes for the next steps:
- The vocabulary has **53** tokens instead of 10. So the embedding table will have 53 rows, and the output will give 53 scores.
- Our sentences use only **10** of them. The other 43 (single letters and half-words like `_hun`) are only there to spell new words. To keep the math readable, the next steps show the 10 used tokens and sum up the rest as "≈ 0%".

<!-- notebook-only -->

The same code is saved in `toddler/bpe.py` as `BPETokenizer`, and the next notebooks import it from there.

<!-- /notebook-only -->

In [ ]:
# Try it: the package version used by the next notebooks gives the same result
from toddler.bpe import BPETokenizer

tok = BPETokenizer.train(CORPUS)
assert tok.merges == merges and tok.vocab == vocab
print(tok.encode("I want milky ."), "→", repr(tok.decode(tok.encode("I want milky ."))))

## 1.6 What this explains about real LLMs

- Tokens are not words. In English, 1 token is about ¾ of a word (about 4 characters).
- Spaces stick to words: "I want" → `I` + ` want`. So `want`, ` want` and ` Want` are 3 different tokens.
- Special tokens mark structure, for example `<|endoftext|>` (end of a document) or the "user" and "assistant" turns in a chat.
- The model sees chunks, not letters. That's why "how many r's in *strawberry*?" used to trip LLMs up.
- Context limits and prices are counted in tokens.
- Real tokenizers start from (or fall back to) the 256 possible bytes, and learn their merges once on a huge amount of text. This gives about 32k–200k tokens.

<!-- notebook-only -->

Optional: the cell below compares with a real tokenizer (`tiktoken`, used by OpenAI models). Install it with `.venv/bin/pip install tiktoken`; the first use also downloads its vocabulary from the internet.

<!-- /notebook-only -->

In [ ]:
# Try it (optional): a real tokenizer
try:
    import tiktoken
    encoding = tiktoken.get_encoding("cl100k_base")
    for text in ["I am hungry. I want milk.", "I wanted milky", "strawberry"]:
        print(f"{text!r:<28} → {[encoding.decode([i]) for i in encoding.encode(text)]}")
except ImportError:
    print("tiktoken is not installed: skip this, or run  .venv/bin/pip install tiktoken")
except Exception as error:
    print("could not load the tokenizer (it needs internet the first time):", error)

## 1.7 The big limit (leads to Step 2)

IDs are just **locker numbers**. They only tell *where* a token sits in the vocabulary, in the order the merges were learned. Two tokens with close IDs are not close in meaning, and two tokens that belong together can have IDs far apart. The model needs numbers that *carry meaning*. That's what embeddings are (Step 2).

<!-- notebook-only -->

## Playground: your own toddler world

Write your own training data in the cell below (**up to 10 lines**), then run the next cell. You see the merges BPE learns, the vocabulary, and how each line is cut into tokens. The next notebooks have the same playground, and there the toddler also trains on your lines.

Ideas to try:
- Add `"I am happy . I want ball ."`: a new letter (`p`), new merges, a bigger vocabulary.
- Set `MAX_MERGES = 10` to stop early, like a real tokenizer with a fixed vocabulary size: rare words stay in pieces.

<!-- /notebook-only -->

In [ ]:
# Playground: your own toddler world (up to 10 lines), then run this cell and the next one
MY_CORPUS = [
    "I am hungry . I want milk .",
    "I am sad . I want mommy .",
    "I am tired . I want bed .",
    "I am happy . I want ball .",
]
MAX_MERGES = None   # None = merge until every word is one token; try 10

In [ ]:
# Playground: train BPE on MY_CORPUS, then cut each line into tokens
from toddler.bpe import BPETokenizer
from toddler.data import check_corpus

my_corpus = check_corpus(MY_CORPUS)
my_tok = BPETokenizer.train(my_corpus, max_merges=MAX_MERGES)
print(f"{len(my_tok.vocab)} tokens = {len(my_tok.vocab) - len(my_tok.merges)} symbols + {len(my_tok.merges)} merges")
print("merges, in order:", " ".join(left + right for left, right in my_tok.merges), "\n")
for line in my_corpus:
    print(f"{line:<30} → {' '.join(my_tok.tokenize(line)):<44} → {my_tok.encode(line)}")

## Your turn

Do each one **by hand first** (paper is fine), using the vocabulary and the merge list above.

### 1.1 Encode "I want bed ." into IDs.

<details>
<summary>Show the answer</summary>
<!-- answer: 21 28 52 22 -->
<!-- format: IDs separated by spaces, e.g. 1 2 3 -->

Each word is one token here: `_I _want _bed _.` → **[21, 28, 52, 22]**.

</details>

In [ ]:
my_answer = None  # a list of IDs, e.g. [1, 2, 3]
check("1.1", my_answer)

### 1.2 How does BPE cut the new word "wand"? Show each step, like the "wanted" table.

<details>
<summary>Show the answer</summary>
<!-- answer: _wan d -->
<!-- format: the final pieces, separated by spaces, e.g. _ab c -->

| merge tried | found? | pieces after |
|---|---|---|
| (start) | | `_ w a n d` |
| #1–#4 | no | `_ w a n d` |
| #5 `_ w` | yes | `_w a n d` |
| #6 `_w a` | yes | `_wa n d` |
| #7 `_wa n` | yes | `_wan d` |
| #8 `_wan t` | **no** (we have `_wan d`) | `_wan d` |
| #9–#32 | no | `_wan d` |

Result: 2 tokens, `_wan` (27) + `d` (17).

</details>

In [ ]:
my_answer = None  # a list of pieces, e.g. ["_ab", "c"]
check("1.2", my_answer)

In [ ]:
# Try it: every step on "wand"
encode_word("_wand", merges, verbose=True, show_misses=True);

### 1.3 Predict the pieces of the new word "tamed". Careful: is there a merge for a plain `a m`?

<details>
<summary>Show the answer</summary>
<!-- answer: _t a m ed -->
<!-- format: the final pieces, separated by spaces, e.g. _ab c -->

`_ t a m e d` → #10 (`e d`) → `_ t a m ed` → #27 (`_ t`) → `_t a m ed`.

`a m` never glues: the merge list only has `_ a` and then `_a m`, that is an `a` right after a word start. Here the `a` comes after `t`.

Result: `_t` `a` `m` `ed` → IDs 47, 2, 3, 30.

</details>

In [ ]:
my_answer = None  # a list of pieces
check("1.3", my_answer)

In [ ]:
# Try it: the merges that fire on "tamed"
encode_word("_tamed", merges, verbose=True);

### 1.4 Decode the IDs `[21, 24, 44, 22]` by hand.

<details>
<summary>Show the answer</summary>
<!-- answer: I am mom . | I am mom -->
<!-- format: the text -->

`21 24 44 22` → `_I _am _mom _.` → `"_I_am_mom_."` → **"I am mom ."**

</details>

In [ ]:
my_answer = None  # a string
check("1.4", my_answer)

### 1.5 Experiment: a smaller vocabulary

Real tokenizers stop at a chosen vocabulary size. Imagine we stop after 10 merges (31 tokens). With this smaller tokenizer, how many tokens is "I am hungry ."?

<details>
<summary>Show the answer</summary>
<!-- answer: 10 | ten -->
<!-- format: a number -->

**10 tokens**: `_I _am _ h u n g r y _.`

`_I`, `_am` and `_.` are built by merges #1 to #4. The merges that build `_hungry` (#11 to #16) are missing, so "hungry" stays in 7 pieces. A bigger vocabulary means fewer, more meaningful pieces.

</details>

In [ ]:
merges_10, vocab_10 = train_bpe(CORPUS, max_merges=10)
print(len(vocab_10), "tokens; merges:", [a + b for a, b in merges_10])

my_answer = None  # a number
check("1.5", my_answer)

In [ ]:
# Try it: the pieces with only 10 merges
print([piece for word in to_words("I am hungry .") for piece in encode_word(word, merges_10)])

### 1.6 Think: `_milk` = 39 is closer to `_hungry` = 36 than to `_mommy` = 46. Does that tell the model anything about meaning? Why?

<details>
<summary>Show the answer</summary>
<!-- choices: *No: an ID is only a row number, given in the order the merges were learned | Yes: close IDs mean close meanings | Yes, but only for words that share letters | It depends on the size of the vocabulary -->

No. An ID is only a row number (a locker number), given in the order the merges were learned. Being close in number means nothing. Meaning comes from the embeddings, in Step 2.

</details>

## Summary

- BPE starts from single symbols and glues the most frequent pair, again and again.
- Training gives a **vocabulary** (token → ID) and a **merge list** (the recipe).
- Encoding applies the merges in order. Decoding glues the pieces back and turns `_` into spaces.
- Common words become one token; rare or new words are cut into known pieces.
- An ID is only a row number: it carries no meaning. Giving it meaning is the job of **Step 2: embeddings**<!-- notebook-only --> (`02_embeddings.ipynb`)<!-- /notebook-only -->.